# EfficientNetB0 evaluation (from scratch)

This notebook loads the **EfficientNetB0 from scratch** model trained in `01_train_efficientnet_b0.ipynb`, evaluates performance on the test set, and displays overall scores + a confusion matrix and a metrics graph.

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b0

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

NUM_CLASSES = 3
class_names = ["NORMAL", "BACTERIA", "VIRUS"]

In [ ]:
%run ../../prepocessing.ipynb

print("\nTaille test_loader :", len(test_loader))

In [ ]:
# Rebuild the EfficientNetB0 model (from scratch) and load the trained weights

model = efficientnet_b0(weights=None)

in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, NUM_CLASSES)

model = model.to(device)

# Checkpoint path saved in 01_train_efficientnet_b0.ipynb
checkpoint_path = "efficientnet_b0_from_scratch_best.pt"

state_dict = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(state_dict)
model.eval()

print(f"Weights loaded from {checkpoint_path}")

In [ ]:
# Evaluation loop on the test set + calculation of scores

all_labels = []
all_preds = []
all_probs = []  # softmax probabilities needed for ROC-AUC

with torch.no_grad():
    for batch in test_loader:
        # In prepocessing.ipynb, each batch is a dict(“image”, “label”)
        images = batch["image"].to(device)
        labels = batch["label"].to(device)

        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        _, preds = torch.max(probs, 1)

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

all_labels = np.array(all_labels)
all_preds = np.array(all_preds)
all_probs = np.array(all_probs)

# Global metrics (macro-average to balance classes)
accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds, average="macro", zero_division=0)
recall = recall_score(all_labels, all_preds, average="macro", zero_division=0)
f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)

# Multiclass ROC-AUC (one-vs-rest)
try:
    roc_auc = roc_auc_score(all_labels, all_probs, multi_class="ovr")
except ValueError:
    roc_auc = float("nan")

print("\nClassification report (per class):")
print(classification_report(all_labels, all_preds, target_names=class_names))

print("\nGlobal metrics (macro-average):")
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1:.4f}")
print(f"ROC-AUC   : {roc_auc:.4f}")

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion matrix - test set (B0 from scratch)")
plt.tight_layout()
plt.show()

# Bar chart of metrics (in %)
metrics_names = ["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"]
metrics_values = [accuracy, precision, recall, f1, roc_auc]

metrics_percent = [m * 100 for m in metrics_values]

plt.figure(figsize=(8, 5))
plt.bar(metrics_names, metrics_percent, color=["#4caf50", "#2196f3", "#ff9800", "#9c27b0", "#f44336"])
plt.ylabel("Score (%)")
plt.ylim(0, 100)
plt.title("Global evaluation metrics (test set) - B0 from scratch")
for i, v in enumerate(metrics_percent):
    plt.text(i, v + 1, f"{v:.1f}%", ha="center", va="bottom")
plt.tight_layout()
plt.show()